In [1]:
# 1. Install Ultralytics and Mount Drive
!pip install ultralytics --quiet
print("✅ Ultralytics installed.")

from google.colab import drive
# Use try-except to avoid re-mounting error if already mounted
try:
    drive.mount('/content/drive')
    print("✅ Google Drive mounted.")
except Exception as e:
    # If the drive is already mounted, this will print the status
    print(f"Drive mount status: Already mounted or encountered error: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.8 MB/s eta 0:00:00
✅ Ultralytics installed.
Mounted at /content/drive
✅ Google Drive mounted.


In [8]:
import os
from ultralytics import YOLO

# --- Configuration ---
BASE_DATA_PATH = '/content/drive/MyDrive/Construction/dataset'
DATA_YAML_PATH = os.path.join(BASE_DATA_PATH, 'data.yaml')
PROJECT_DIR = '/content/drive/MyDrive/Construction/yolov8_runs'
EXPERIMENT_NAME = 'yolov8s_construction_run_18' # Incrementing run number

# Class names (assuming you are using 10 classes, as required by your labels 0-9)
CLASS_NAMES = [
    'HardHat', 'SafetyVest', 'Person', 'Excavator', 'Ladder',
    'Barricade', 'Tool', 'Vehicle', 'Gloves', 'SafetyGlasses'
]

# --- CRITICAL FIX 1: Create the data.yaml file with the CORRECTED paths (no 'test') ---
yaml_content = f"""
path: {BASE_DATA_PATH}
train: train/images
val: val/images
# Removed: test: test/images
# Define the number of classes and their names
nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""

with open(DATA_YAML_PATH, 'w') as f:
    f.write(yaml_content)

print(f"\n✅ Configuration file updated (removed 'test' path): {DATA_YAML_PATH}")
print(f"✅ Class Count (nc) confirmed to be: {len(CLASS_NAMES)}")


# --- CRITICAL FIX 2: Delete the corrupted cache files (for train and val) ---
cache_files = [
    os.path.join(BASE_DATA_PATH, 'train', 'labels.cache'),
    os.path.join(BASE_DATA_PATH, 'val', 'labels.cache')
]

deleted_count = 0
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✅ Deleted old cache file: {cache_file}")
        deleted_count += 1

print(f"\nSuccessfully deleted {deleted_count} cache files. Rerunning training to force a fresh dataset scan.")


# 1. Load a pre-trained YOLOv8s model
model = YOLO('yolov8s.pt')

print(f"\nStarting fresh training on model: YOLOv8s")

# 2. Start Retraining/Fine-Tuning
results = model.train(
    data=DATA_YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    name=EXPERIMENT_NAME,
    project=PROJECT_DIR,
    optimizer='AdamW',
    lr0=0.001,
)

print(f"\n✅ Training command initiated. Results will be saved to: {os.path.join(PROJECT_DIR, EXPERIMENT_NAME)}")


✅ Configuration file updated (removed 'test' path): /content/drive/MyDrive/Construction/dataset/data.yaml
✅ Class Count (nc) confirmed to be: 10
✅ Deleted old cache file: /content/drive/MyDrive/Construction/dataset/train/labels.cache
✅ Deleted old cache file: /content/drive/MyDrive/Construction/dataset/val/labels.cache

Successfully deleted 2 cache files. Rerunning training to force a fresh dataset scan.

Starting fresh training on model: YOLOv8s
Ultralytics 8.3.227 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Construction/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_o

In [9]:
import os
from ultralytics import YOLO

# --- Configuration for Construction Detection Project ---
# Base directory where the training results were saved
PROJECT_DIR = '/content/drive/MyDrive/Construction/yolov8_runs'
# The specific folder name for the completed run
EXPERIMENT_NAME = 'yolov8s_construction_run_18'
# The data configuration file used for this training
DATA_YAML_PATH = '/content/drive/MyDrive/Construction/dataset/data.yaml'

# Construct the full path to the best trained weights
BEST_WEIGHTS_PATH = os.path.join(
    PROJECT_DIR,
    EXPERIMENT_NAME,
    'weights',
    'best.pt'
)

print(f"Loading weights from: {BEST_WEIGHTS_PATH}")

try:
    # Load the best trained model (yolov8s_construction_run_18/weights/best.pt)
    model = YOLO(BEST_WEIGHTS_PATH)

    # Run validation on the validation set
    metrics = model.val(
        data=DATA_YAML_PATH,  # Use the construction data configuration
        split='val',          # Explicitly target the validation set
        imgsz=640             # Use the same image size as training
    )

    # Print the validation results
    print("\n✅ Validation Complete (Construction Detection):")
    print(f"mAP50: {metrics.results_dict['metrics/mAP50(B)']:.4f}")
    print(f"mAP50-95: {metrics.results_dict['metrics/mAP50-95(B)']:.4f}")

except Exception as e:
    print(f"\n❌ ERROR: Could not run validation.")
    print(f"Detailed Error: {e}")

Loading weights from: /content/drive/MyDrive/Construction/yolov8_runs/yolov8s_construction_run_18/weights/best.pt
Ultralytics 8.3.227 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 11,129,454 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 4.1±8.1 ms, read: 26.5±22.0 MB/s, size: 59.5 KB)
val: Scanning /content/drive/MyDrive/Construction/dataset/val/labels.cache... 114 images, 10 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 114/114 186.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.6it/s 5.0s
                   all        114        697      0.904      0.767      0.835      0.546
               HardHat         42         79      0.942      0.797      0.916      0.627
            SafetyVest         19         21          1      0.943      0.961      0.698
                Person         37         69      0.891      0.592      0.728      0.411


In [14]:
import os
from ultralytics import YOLO

# --- Configuration (FIXED for Construction Detection Project) ---
PROJECT_DIR = '/content/drive/MyDrive/Construction/yolov8_runs'
EXPERIMENT_NAME = 'yolov8s_construction_run_18'
DATA_YAML_PATH = '/content/drive/MyDrive/Construction/dataset/data.yaml'

BEST_WEIGHTS_PATH = os.path.join(
    PROJECT_DIR,
    EXPERIMENT_NAME,
    'weights',
    'best.pt'
)

# FIX: Pointing to the existing 'val' images since 'test' does not exist
SOURCE_DATA = '/content/drive/MyDrive/Construction/dataset/val/images'

# Initialize results to None to handle prediction failures cleanly
results = None

# --- 1. Load Model ---
print(f"Loading model from: {BEST_WEIGHTS_PATH}")
try:
    model = YOLO(BEST_WEIGHTS_PATH, verbose=False)
    print("✅ Model loaded successfully.")
except Exception as e:
    print(f"\n❌ FATAL ERROR: Could not load model. Detailed Error: {e}")
    exit()

# --- 2. Run Inference (Prediction & Visuals on VAL Images) ---
print(f"\nStarting visual inference (prediction) on validation images in: {SOURCE_DATA}")
try:
    # Running prediction will save annotated images and return results objects
    results = model.predict(
        source=SOURCE_DATA,
        save=True,
        conf=0.25,   # Minimum confidence threshold
        iou=0.7,     # NMS threshold
        project=PROJECT_DIR,
        name='val_predictions_run', # Changed name to reflect 'val' data is used
        exist_ok=True,
        verbose=False
    )
    print("\n✅ Visual Inference (Prediction) Complete.")
    print(f"Check your visual predictions in: {os.path.join(PROJECT_DIR, 'val_predictions_run')}")
except Exception as e:
    print(f"\n❌ ERROR during inference: {e}")

# --------------------------------------------------------------------------------------
# --- 3. Detect FPS and Latency from Prediction Results ---
# --------------------------------------------------------------------------------------
print("\n--- Performance Metrics (From Prediction Run) ---")
# Check if results are valid (not None) and contain speed data
if results and hasattr(results[0], 'speed') and results[0].speed:
    speed_ms = results[0].speed

    inference_ms = speed_ms.get('inference', 0)
    total_ms = sum(speed_ms.values())

    core_inference_fps = (1000 / inference_ms) if inference_ms > 0 else 0
    total_e2e_fps = (1000 / total_ms) if total_ms > 0 else 0

    print(f"Core Inference Latency: {inference_ms:.2f} ms")
    print(f"Total End-to-End Latency: {total_ms:.2f} ms")
    print(f"Core Inference FPS: {core_inference_fps:.2f} FPS")
    print(f"Total End-to-End FPS: {total_e2e_fps:.2f} FPS")
else:
    # This warning is expected if no images were found or prediction failed
    print("Warning: Prediction failed or could not retrieve speed metrics.")

# --------------------------------------------------------------------------------------
# --- 4. Calculate Test Accuracy (Quantitative Validation on VAL Split) ---
# --------------------------------------------------------------------------------------
print("\nStarting quantitative validation (Accuracy calculation) on the VAL split...")
try:
    # FIX: Target the designated 'val' set, as 'test' does not exist
    test_metrics = model.val(
        data=DATA_YAML_PATH,
        split='val',
        imgsz=640,
        verbose=False
    )

    # Print the Validation Accuracy results
    print("\n✅ Validation Accuracy Calculation Complete:")
    print(f"mAP50: {test_metrics.results_dict['metrics/mAP50(B)']:.4f}")
    print(f"mAP50-95: {test_metrics.results_dict['metrics/mAP50-95(B)']:.4f}")

except Exception as e:
    print(f"\n❌ ERROR during validation accuracy calculation.")
    print(f"Detailed Error: {e}")

Loading model from: /content/drive/MyDrive/Construction/yolov8_runs/yolov8s_construction_run_18/weights/best.pt
✅ Model loaded successfully.

Starting visual inference (prediction) on validation images in: /content/drive/MyDrive/Construction/dataset/val/images
Results saved to /content/drive/MyDrive/Construction/yolov8_runs/val_predictions_run

✅ Visual Inference (Prediction) Complete.
Check your visual predictions in: /content/drive/MyDrive/Construction/yolov8_runs/val_predictions_run

--- Performance Metrics (From Prediction Run) ---
Core Inference Latency: 18.31 ms
Total End-to-End Latency: 27.98 ms
Core Inference FPS: 54.63 FPS
Total End-to-End FPS: 35.74 FPS

Starting quantitative validation (Accuracy calculation) on the VAL split...
Ultralytics 8.3.227 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 34.6±17.4 MB/s, size: 60.3 KB)
val: Scanning /content/drive/MyDrive/Construction/dataset/val/labels.cache... 114 image